# Edge Perturbation Analysis

Two scenarios from supervisor suggestion:

**Scenario 1 — Edge Count vs Performance**  
x-axis: total edge count (GT-delta to GT+delta)  
y-axis: task accuracy + CCI  
Each count: 5 random graphs → boxplot

**Scenario 2 — Single-Edge Perturbation**  
x-axis: CCI of model trained on perturbed graph  
y-axis: accuracy  
Each dot: one edge deleted or added


In [ ]:
import pandas as pd, numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
import warnings; warnings.filterwarnings('ignore')

sns.set_theme(style='whitegrid', font_scale=1.1)
plt.rcParams['figure.dpi'] = 120
plt.rcParams['axes.spines.top'] = False
plt.rcParams['axes.spines.right'] = False

EXPERIMENTS_ROOT = Path('/home/dani00003/mCREAM/experiments')
DAG_CFMNIST      = Path('/home/dani00003/mCREAM/data/FashionMNIST/Complete_Concept_FMNIST_DAG.csv')
DATASETS = ['Complete_Concept_FMNIST', 'CelebA']
MODEL_MAP = {
    'Complete_Concept_FMNIST': 'Standard_FashionMNIST',
    'CelebA':                  'Standard_CelebA',
}
print(f'Root exists: {EXPERIMENTS_ROOT.exists()}')


In [ ]:
# =========================================================================
# Visualise perturbed graphs: rows = delta, cols = seeds
# Shows exactly which edges changed per graph
# =========================================================================
def plot_perturbed_graphs(dataset_key, graph_type, dag_path, K, T, delta=5, n_seeds=5):
    import pandas as pd, numpy as np, torch
    from pathlib import Path

    data_dir = Path(f'/home/dani00003/mCREAM/data/{"FashionMNIST" if dataset_key=="cfmnist" else "CelebA"}/edge_count_experiments')
    gt_df    = pd.read_csv(dag_path, index_col=0)
    node_names = list(gt_df.index)
    concepts   = node_names[:K]
    tasks      = node_names[K:]
    gt_vals    = (gt_df.values != 0).astype(int)

    if graph_type == 'u2c':
        gt_block = gt_vals[:K, :K]
        row_labels, col_labels = concepts, concepts
        gt_count = gt_block.sum()
    else:
        gt_block = gt_vals[K:, :K]
        row_labels, col_labels = tasks, concepts
        gt_count = gt_block.sum()

    counts = list(range(max(0, gt_count - delta), gt_count + delta + 1))
    n_rows = len(counts)
    n_cols = n_seeds + 1   # +1 for GT reference

    fig, axes = plt.subplots(n_rows, n_cols,
                              figsize=(2.2*n_cols, 2.0*n_rows),
                              squeeze=False)
    fig.suptitle(f'{dataset_key.upper()} — {graph_type.upper()} perturbed graphs
'
                 f'Rows = edge count (GT={gt_count}), Cols = seeds (right = GT reference)',
                 fontsize=11, fontweight='bold')

    for row_i, count in enumerate(counts):
        diff = count - gt_count
        diff_str = f'+{diff}' if diff > 0 else str(diff)
        label = f'{diff_str}  ({count} edges)'
        if diff == 0:
            label = f'GT  ({count} edges)'

        for seed in range(n_seeds):
            ax = axes[row_i, seed]
            fname = data_dir / f'edge_count_{graph_type}_{count}edges_seed{seed}.csv'
            if not fname.exists():
                ax.text(0.5, 0.5, 'not found', ha='center', va='center',
                        transform=ax.transAxes, fontsize=7, color='red')
                ax.axis('off')
                continue
            perturbed_df = pd.read_csv(fname, index_col=0)
            p_vals = (perturbed_df.values != 0).astype(int)
            if graph_type == 'u2c':
                p_block = p_vals[:K, :K]
            else:
                p_block = p_vals[K:, :K]

            # Show: GT=blue, added=green, deleted=red
            display_mat = np.zeros((*p_block.shape, 3))
            for r in range(p_block.shape[0]):
                for c in range(p_block.shape[1]):
                    if gt_block[r,c]==1 and p_block[r,c]==1:   # kept
                        display_mat[r,c] = [0.2, 0.4, 0.8]    # blue
                    elif gt_block[r,c]==0 and p_block[r,c]==1: # added (spurious)
                        display_mat[r,c] = [0.2, 0.8, 0.2]    # green
                    elif gt_block[r,c]==1 and p_block[r,c]==0: # deleted
                        display_mat[r,c] = [0.9, 0.2, 0.2]    # red
                    else:                                       # absent
                        display_mat[r,c] = [0.95, 0.95, 0.95] # light grey

            ax.imshow(display_mat, aspect='auto', interpolation='nearest')
            ax.set_xticks([]); ax.set_yticks([])
            if seed == 0:
                ax.set_ylabel(label, fontsize=6.5, rotation=0, ha='right',
                              labelpad=60, va='center')
            ax.set_title(f's{seed}', fontsize=7)

        # Last col: GT reference
        ax_gt = axes[row_i, n_seeds]
        display_gt = np.zeros((*gt_block.shape, 3))
        for r in range(gt_block.shape[0]):
            for c in range(gt_block.shape[1]):
                if gt_block[r,c]==1: display_gt[r,c] = [0.2, 0.4, 0.8]
                else:                display_gt[r,c] = [0.95, 0.95, 0.95]
        ax_gt.imshow(display_gt, aspect='auto', interpolation='nearest')
        ax_gt.set_xticks([]); ax_gt.set_yticks([])
        ax_gt.set_title('GT', fontsize=7, fontweight='bold')
        if diff == 0:
            for spine in ax_gt.spines.values():
                spine.set_edgecolor('gold'); spine.set_linewidth(3)

    # Legend
    from matplotlib.patches import Patch
    legend_elements = [
        Patch(facecolor=[0.2,0.4,0.8], label='Edge kept'),
        Patch(facecolor=[0.9,0.2,0.2], label='Edge deleted'),
        Patch(facecolor=[0.2,0.8,0.2], label='Edge added (spurious)'),
        Patch(facecolor=[0.95,0.95,0.95], label='No edge'),
    ]
    fig.legend(handles=legend_elements, loc='lower center',
               ncol=4, fontsize=8, bbox_to_anchor=(0.5, -0.01))

    plt.tight_layout(rect=[0.08, 0.03, 1, 1])
    plt.savefig(f'perturbed_graphs_{dataset_key}_{graph_type}.png',
                dpi=150, bbox_inches='tight')
    plt.show()


# Run for whatever graph types exist
import os
data_base = Path('/home/dani00003/mCREAM/data/FashionMNIST/edge_count_experiments')
if any(data_base.glob('edge_count_u2c_*.csv')):
    plot_perturbed_graphs('cfmnist', 'u2c', DAG_CFMNIST, K=11, T=10, delta=5, n_seeds=5)
if any(data_base.glob('edge_count_c2y_*.csv')):
    plot_perturbed_graphs('cfmnist', 'c2y', DAG_CFMNIST, K=11, T=10, delta=5, n_seeds=5)


## 1. Load Results


In [ ]:
def load_edge_results(root, exp_folder):
    """Load last_metrics CSVs from an edge perturbation experiment folder."""
    rows = []
    for dataset, model in MODEL_MAP.items():
        md = root / dataset / 'train_cbm' / model / exp_folder / 'last_metrics'
        if not md.exists(): print(f'  [SKIP] {md}'); continue
        for f in sorted(md.glob('*.csv')):
            try:
                df = pd.read_csv(f)
                if len(df) == 0: continue
                df['dataset']  = dataset
                df['exp_name'] = f.stem
                rows.append(df)
            except Exception as e:
                print(f'  ERR {f}: {e}')
    if not rows: print(f'No results in {exp_folder}'); return pd.DataFrame()
    return pd.concat(rows, ignore_index=True)

# Scenario 1: edge_count_u2c_{N}edges_seed{s}
count_df = load_edge_results(EXPERIMENTS_ROOT, 'edge_count_experiments')
if len(count_df) > 0:
    # Parse edge count and seed from experiment name
    count_df['graph_type'] = count_df['exp_name'].str.extract(r'edge_count_(u2c|c2y)_')[0]
    count_df['edge_count'] = count_df['exp_name'].str.extract(r'_(\d+)edges_')[0].astype(int)
    print(f'  graph_types: {sorted(count_df["graph_type"].dropna().unique())}')
    count_df['seed']       = count_df['exp_name'].str.extract(r'_seed(\d+)$').astype(int)
    print(f'Loaded edge count results: {len(count_df)} rows')
    print(f'  Edge counts: {sorted(count_df["edge_count"].unique())}')

# Scenario 2: del_edge_* and add_edge_*
perturb_df = load_edge_results(EXPERIMENTS_ROOT, 'single_edge_perturbation')
if len(perturb_df) > 0:
    perturb_df['perturb_type'] = perturb_df['exp_name'].str[:3]  # 'del' or 'add'
    perturb_df['edge_label']   = perturb_df['exp_name'].str[4:]  # edge name
    print(f'Loaded single-edge perturbation results: {len(perturb_df)} rows')
    print(f'  Deletions: {(perturb_df["perturb_type"]=="del").sum()}')
    print(f'  Additions: {(perturb_df["perturb_type"]=="add").sum()}')


## 2. GT Baseline Reference


In [ ]:
# Load GT CREAM result for reference line
GT_EXP = {'Complete_Concept_FMNIST': 'CREAM_best_cfmnist',
           'CelebA':                  'CREAM_best_celeba'}
gt_rows = []
for dataset, model in MODEL_MAP.items():
    md = EXPERIMENTS_ROOT / dataset / 'train_cbm' / model / GT_EXP.get(dataset,'') / 'last_metrics'
    if not md.exists(): continue
    for f in sorted(md.glob('*.csv')):
        df = pd.read_csv(f); df['dataset'] = dataset; gt_rows.append(df)
gt_df = pd.concat(gt_rows, ignore_index=True) if gt_rows else pd.DataFrame()
if len(gt_df) > 0:
    for ds in gt_df['dataset'].unique():
        r = gt_df[gt_df['dataset']==ds].iloc[0]
        print(f'GT {ds}: acc={r.get("test_task_accuracy",float("nan")):.4f}  CCI={r.get("CCI","N/A")}')


if len(count_df) == 0:
    print('No edge count results yet.')
else:
    for dataset in DATASETS:
        ds = count_df[count_df['dataset']==dataset]
        if ds.empty: continue

        graph_types_present = sorted(ds['graph_type'].dropna().unique())
        n_types = len(graph_types_present)
        TYPE_COLOR = {'u2c': '#3498db', 'c2y': '#e74c3c'}
        GT_COUNTS  = {'u2c': 17, 'c2y': 19}  # cfmnist GT edge counts

        for metric, ylabel in [('test_task_accuracy','Task Accuracy'), ('CCI','CCI')]:
            if metric not in ds.columns: continue

            fig, axes = plt.subplots(1, n_types, figsize=(7*n_types, 5), sharey=False)
            if n_types == 1: axes = [axes]
            fig.suptitle(f'{dataset}
Scenario 1: Edge Count vs {ylabel}
'
                         f'Each box = 5 random graphs with that many edges',
                         fontsize=12, fontweight='bold')

            for ax, gtype in zip(axes, graph_types_present):
                sub = ds[ds['graph_type']==gtype].dropna(subset=[metric])
                if sub.empty: ax.set_title(f'{gtype} no data'); continue

                edge_counts = sorted(sub['edge_count'].unique())
                data_per_count = [sub[sub['edge_count']==ec][metric].values
                                  for ec in edge_counts]

                bp = ax.boxplot(data_per_count, positions=edge_counts,
                                widths=0.5, patch_artist=True,
                                medianprops=dict(color='black', lw=2))
                color = TYPE_COLOR.get(gtype, '#3498db')
                for patch in bp['boxes']:
                    patch.set_facecolor(color); patch.set_alpha(0.7)

                # GT reference line (accuracy)
                if len(gt_df) > 0 and metric in gt_df.columns:
                    sub2 = gt_df[gt_df['dataset']==dataset]
                    if not sub2.empty and pd.notna(sub2[metric].mean()):
                        v = sub2[metric].mean()
                        ax.axhline(v, color='black', lw=2, ls='--',
                                   label=f'GT CREAM: {v:.4f}')

                # Vertical red line at GT edge count
                gt_n = GT_COUNTS.get(gtype, edge_counts[len(edge_counts)//2])
                if gt_n in edge_counts:
                    ax.axvline(x=gt_n, color='red', ls=':', lw=2, alpha=0.8,
                               label=f'GT count ({gt_n} edges)')

                ax.set_xlabel('Number of edges in graph', fontsize=10)
                ax.set_ylabel(ylabel, fontsize=10)
                ax.set_title(f'{gtype.upper()} ({"concept->concept" if gtype=="u2c" else "concept->task"})',
                             fontsize=10)
                ax.set_xticks(edge_counts)
                ax.tick_params(labelsize=8)
                ax.legend(fontsize=8, framealpha=0.9)

            plt.tight_layout()
            plt.savefig(f'edge_count_{metric}_{dataset}.png', dpi=150, bbox_inches='tight')
            plt.show()


In [ ]:
# Line plot: each line = one seed (one specific random graph)
# x = edge count, y = accuracy / CCI
# Helps identify WHICH random graph (seed) performs best/worst consistently
if len(count_df) == 0:
    print('No results yet.')
else:
    SEED_COLORS  = ['#1f77b4','#d62728','#2ca02c','#ff7f0e','#9467bd']
    SEED_MARKERS = ['o','s','^','D','v']

    for dataset in DATASETS:
        ds = count_df[count_df['dataset']==dataset]
        if ds.empty: continue

        graph_types_present = sorted(ds['graph_type'].dropna().unique())

        for gtype in graph_types_present:
            sub = ds[ds['graph_type']==gtype]
            if sub.empty: continue

            edge_counts = sorted(sub['edge_count'].unique())
            seeds       = sorted(sub['seed'].unique())
            GT_N        = {'u2c': 17, 'c2y': 19}.get(gtype, edge_counts[len(edge_counts)//2])

            fig, axes = plt.subplots(1, 2, figsize=(13, 5))
            fig.suptitle(
                f'{dataset} — {gtype.upper()} ({"concept->concept" if gtype=="u2c" else "concept->task"})
'
                f'Per-seed line plot: each line = one random graph
'
                f'Find which graph (seed) is consistently best/worst',
                fontsize=11, fontweight='bold'
            )

            for ax, metric, ylabel in [
                (axes[0], 'test_task_accuracy', 'Task Accuracy'),
                (axes[1], 'CCI',               'CCI'),
            ]:
                if metric not in sub.columns:
                    ax.set_title(f'{ylabel} — not found'); continue

                for s_idx, seed in enumerate(seeds):
                    seed_data = sub[sub['seed']==seed].sort_values('edge_count')
                    if seed_data.empty: continue
                    ax.plot(
                        seed_data['edge_count'],
                        seed_data[metric],
                        color=SEED_COLORS[s_idx % len(SEED_COLORS)],
                        marker=SEED_MARKERS[s_idx % len(SEED_MARKERS)],
                        markersize=7, linewidth=2,
                        label=f'Seed {seed}'
                    )

                # GT CREAM reference
                if len(gt_df) > 0 and metric in gt_df.columns:
                    gt_sub = gt_df[gt_df['dataset']==dataset]
                    if not gt_sub.empty and pd.notna(gt_sub[metric].mean()):
                        v = gt_sub[metric].mean()
                        ax.axhline(v, color='black', lw=2.5, ls='--',
                                   label=f'GT CREAM: {v:.4f}', zorder=10)

                # GT edge count vertical line
                ax.axvline(x=GT_N, color='red', ls=':', lw=2, alpha=0.7,
                           label=f'GT count ({GT_N} edges)')

                ax.set_xlabel('Number of edges in graph', fontsize=10)
                ax.set_ylabel(ylabel, fontsize=10)
                ax.set_title(ylabel, fontsize=10)
                ax.set_xticks(edge_counts)
                ax.legend(fontsize=8, framealpha=0.9)
                ax.tick_params(labelsize=8)

            plt.tight_layout()
            plt.savefig(f'edge_count_lineplot_{dataset}_{gtype}.png',
                        dpi=150, bbox_inches='tight')
            plt.show()


In [ ]:
if len(count_df) == 0:
    print('No edge count results yet.')
else:
    for dataset in DATASETS:
        ds = count_df[count_df['dataset']==dataset]
        if ds.empty: continue

        # GT edge count for vertical reference line
        gt_count = ds[ds['edge_count'] == ds['edge_count'].median()]['edge_count'].iloc[0] if len(ds) > 0 else None
        # Better: find GT count from config name (seed0 = GT if diff==0)

        fig, axes = plt.subplots(1, 2, figsize=(13, 5))
        fig.suptitle(f'{dataset}\nScenario 1: Edge Count vs Performance\n'
                     f'Each box = 5 random graphs with that many edges',
                     fontsize=12, fontweight='bold')

        for ax, metric, ylabel in [
            (axes[0], 'test_task_accuracy', 'Task Accuracy'),
            (axes[1], 'CCI',               'CCI'),
        ]:
            if metric not in ds.columns: ax.set_title(f'{ylabel} — not found'); continue

            sub = ds.dropna(subset=[metric])
            # Boxplot: x = edge count, distribution over seeds
            edge_counts = sorted(sub['edge_count'].unique())
            data_per_count = [sub[sub['edge_count']==ec][metric].values for ec in edge_counts]

            bp = ax.boxplot(data_per_count, positions=edge_counts,
                            widths=0.5, patch_artist=True,
                            medianprops=dict(color='black', lw=2))
            for patch in bp['boxes']: patch.set_facecolor('#3498db'); patch.set_alpha(0.7)

            # GT reference line
            if len(gt_df) > 0 and metric in gt_df.columns:
                sub2 = gt_df[gt_df['dataset']==dataset]
                if not sub2.empty and pd.notna(sub2[metric].mean()):
                    v = sub2[metric].mean()
                    ax.axhline(v, color='black', lw=2, ls='--', label=f'GT: {v:.4f}')
                    ax.legend(fontsize=9)

            # Vertical line at GT edge count
            # GT is the count where NO edges were added/removed
            # Find it: the count present in all seeds with identical graphs
            ax.axvline(x=edge_counts[len(edge_counts)//2],
                       color='red', ls=':', lw=1.5, alpha=0.6, label='GT count')

            ax.set_xlabel('Number of edges in graph', fontsize=10)
            ax.set_ylabel(ylabel, fontsize=10)
            ax.set_title(ylabel, fontsize=10)
            ax.set_xticks(edge_counts)
            ax.tick_params(labelsize=8)

        plt.tight_layout()
        plt.savefig(f'edge_count_vs_performance_{dataset}.png', dpi=150, bbox_inches='tight')
        plt.show()


## 4. Scenario 2 — Single-Edge Perturbation: CCI vs Accuracy Scatter


In [ ]:
if len(perturb_df) == 0:
    print('No single-edge perturbation results yet.')
else:
    for dataset in DATASETS:
        ds = perturb_df[perturb_df['dataset']==dataset]
        if ds.empty: continue
        if 'CCI' not in ds.columns or 'test_task_accuracy' not in ds.columns:
            print(f'{dataset}: CCI or accuracy not found'); continue

        fig, ax = plt.subplots(figsize=(9, 6))
        fig.suptitle(f'{dataset}\nScenario 2: Single-Edge Perturbation\n'
                     f'Each dot = one edge deleted/added, model retrained',
                     fontsize=12, fontweight='bold')

        # Deletions
        del_d = ds[ds['perturb_type']=='del'].dropna(subset=['CCI','test_task_accuracy'])
        if not del_d.empty:
            ax.scatter(del_d['CCI'], del_d['test_task_accuracy'],
                       color='#e74c3c', s=80, alpha=0.8, zorder=5,
                       label=f'Deletion ({len(del_d)} edges)')
            # Label most impactful edge (lowest accuracy)
            worst = del_d.nsmallest(3, 'test_task_accuracy')
            for _, row in worst.iterrows():
                label = row['edge_label'].replace('edge_','').replace('_',' ')
                ax.annotate(label, (row['CCI'], row['test_task_accuracy']),
                            fontsize=7, xytext=(5,3), textcoords='offset points')

        # Additions
        add_d = ds[ds['perturb_type']=='add'].dropna(subset=['CCI','test_task_accuracy'])
        if not add_d.empty:
            ax.scatter(add_d['CCI'], add_d['test_task_accuracy'],
                       color='#2ecc71', s=80, alpha=0.8, marker='^', zorder=5,
                       label=f'Addition ({len(add_d)} edges)')

        # GT reference point
        if len(gt_df) > 0:
            sub2 = gt_df[gt_df['dataset']==dataset]
            if not sub2.empty:
                gt_acc = sub2['test_task_accuracy'].mean()
                gt_cci = sub2['CCI'].mean() if 'CCI' in sub2.columns else None
                if gt_cci is not None:
                    ax.scatter([gt_cci], [gt_acc], color='black', s=200,
                               marker='*', zorder=10, label=f'GT ({gt_cci:.3f}, {gt_acc:.4f})')

        ax.set_xlabel('CCI (concept channel importance)', fontsize=10)
        ax.set_ylabel('Task Accuracy', fontsize=10)
        ax.axhline(0.5, color='red', ls=':', alpha=0.4, lw=1)
        ax.legend(fontsize=9, framealpha=0.9)
        ax.tick_params(labelsize=8)
        plt.tight_layout()
        plt.savefig(f'single_edge_perturbation_{dataset}.png', dpi=150, bbox_inches='tight')
        plt.show()


## 5. Edge Importance Ranking (Deletion)


In [ ]:
if len(perturb_df) == 0:
    print('No results yet.')
else:
    for dataset in DATASETS:
        ds = perturb_df[
            (perturb_df['dataset']==dataset) &
            (perturb_df['perturb_type']=='del')
        ].copy()
        if ds.empty or 'test_task_accuracy' not in ds.columns: continue

        # GT accuracy for delta computation
        gt_acc = None
        if len(gt_df) > 0:
            sub2 = gt_df[gt_df['dataset']==dataset]
            if not sub2.empty: gt_acc = sub2['test_task_accuracy'].mean()

        ds['acc_drop'] = (gt_acc - ds['test_task_accuracy']) if gt_acc else ds['test_task_accuracy']
        ds['cci_drop'] = (gt_df[gt_df['dataset']==dataset]['CCI'].mean() - ds['CCI']) if (len(gt_df)>0 and 'CCI' in ds.columns) else ds['CCI']
        ds['edge_name'] = ds['edge_label'].str.replace('edge_','').str.replace('_',' → ',n=1)

        ds_sorted = ds.sort_values('acc_drop', ascending=False)

        fig, axes = plt.subplots(1, 2, figsize=(13, max(4, len(ds)*0.35)))
        fig.suptitle(f'{dataset} — Edge Importance by Accuracy Drop\n'
                     f'Removing which single edge hurts most?',
                     fontsize=12, fontweight='bold')

        for ax, col, xlabel in [
            (axes[0], 'acc_drop', 'Accuracy drop (GT - perturbed)'),
            (axes[1], 'cci_drop', 'CCI drop (GT - perturbed)'),
        ]:
            if col not in ds_sorted.columns: continue
            colors = ['#e74c3c' if v > 0 else '#3498db' for v in ds_sorted[col]]
            ax.barh(ds_sorted['edge_name'], ds_sorted[col], color=colors, alpha=0.8)
            ax.axvline(0, color='black', lw=1)
            ax.set_xlabel(xlabel, fontsize=9)
            ax.tick_params(labelsize=8)
            ax.set_title(xlabel.split('(')[0].strip(), fontsize=10)

        plt.tight_layout()
        plt.savefig(f'edge_importance_ranking_{dataset}.png', dpi=150, bbox_inches='tight')
        plt.show()
